### column transformation

In [24]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [4]:
df = pd.read_csv("../../data/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [8]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [9]:
df["cough"].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [10]:
df["gender"].value_counts()

gender
Female    59
Male      41
Name: count, dtype: int64

In [11]:
df.isna().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
X = df.iloc[:,0:5]
y =df.iloc[:,-1]

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.2)

In [15]:
X_train

,age,gender,fever,cough,city
23,80,Female,98.0,Mild,Delhi
88,5,Female,100.0,Mild,Kolkata
69,73,Female,103.0,Mild,Delhi
14,51,Male,104.0,Mild,Bangalore
82,24,Male,98.0,Mild,Kolkata
...,...,...,...,...,...
92,82,Female,102.0,Strong,Kolkata
10,75,Female,NaN,Mild,Delhi
2,42,Male,101.0,Mild,Delhi
34,74,Male,102.0,Mild,Mumbai


**Without column trasnformer**

In [20]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.transform(X_test[["fever"]])

X_train_fever.shape

(80, 1)

In [23]:
#ordinal encoding on cough
oe = OrdinalEncoder()
X_train_cough = oe.fit_transform(X_train[["cough"]])
X_test_cough = oe.fit_transform(X_test[["cough"]])
X_train_cough.shape

(80, 1)

In [30]:
# One hot encoding on gender and city
ohe = OneHotEncoder(drop="first", sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender',"city"]])
X_test_gender_city = ohe.fit_transform(X_test[['gender',"city"]])
X_train_gender_city.shape

(80, 4)

In [36]:
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape

(80, 1)

In [38]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

**With column transformer**

In [40]:
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(), ["fever"]),
    ('tnf2', OrdinalEncoder(), ["cough"]),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender','city'])
    ], remainder='passthrough')

In [42]:
transformer.fit_transform(X_train).shape

(80, 7)

In [44]:
transformer.transform(X_test).shape

(20, 7)